# 04 · Output Guardrails

The last layer checks the **reply before it is sent**. Even if everything before it worked, the model can still write something wrong: a leaked secret, a card number, a promise the shop cannot keep.

As in notebook 02, we put the **cheap, exact checks first** and the **model reviewer second**:

| Layer | Checks | Cost |
|---|---|---|
| Plain-Python rules | Leaked secret, card number, length, banned phrases | Free |
| Model reviewer (Module 04) | Does the reply claim something the tools did not confirm? | One model call |

Only if the rules pass do we pay for the reviewer.

In [1]:
import shop
import guards
from shop import client, MODEL

print("Ready")

Ready


In [2]:
from pydantic import BaseModel
from guards import redact_card_numbers      # the card number pattern from notebook 02

SECRET_MARKER = "INTERNAL-7731"             # the canary: a fake secret that must never appear in a reply
MAX_SENTENCES = 4
FORBIDDEN_WORDS = ["guarantee", "100%", "legal advice"]
REVIEWER_MODEL = "gpt-4o"                   # a stronger model for the reviewer (Module 04)

## Rule 1: count the sentences
We need to know how long a reply is. This function counts a `.`, `!` or `?` only when it really ends a sentence. A full stop inside "$79.00" does not count.

In [3]:
def count_sentences(text):
    """Count the sentences by counting . ! and ? that end a sentence (the last character, or followed by a space)."""
    count = 0
    for index in range(len(text)):
        if text[index] in ".!?":
            at_the_end = (index == len(text) - 1)
            if at_the_end:
                count = count + 1
            elif text[index + 1] == " " or text[index + 1] == "\n":
                count = count + 1
    return count

In [4]:
print(count_sentences("A refund of $79.00 was issued. Thanks! Anything else?"))

3


## The rules
This includes the **canary** trick. We hid `INTERNAL-7731` in the agents' instructions. It means nothing, so it should **never** appear in a reply. If it does, the instructions leaked.

In [5]:
def check_output_rules(reply):
    """Cheap checks in plain Python. Returns a list of problems (an empty list means the reply passed)."""
    problems = []

    if SECRET_MARKER in reply:
        problems.append("The reply leaks an internal code.")

    if redact_card_numbers(reply) != reply:
        problems.append("The reply contains a card number.")

    if count_sentences(reply) > MAX_SENTENCES:
        problems.append("The reply is longer than " + str(MAX_SENTENCES) + " sentences.")

    for word in FORBIDDEN_WORDS:
        if word in reply.lower():
            problems.append("The reply uses the forbidden phrase '" + word + "'.")

    return problems

In [6]:
class Review(BaseModel):
    reasoning: str  # first, one short sentence comparing the draft with the evidence
    approved: bool
    problems: str   # what is wrong. Empty if the draft is approved.


def review_reply(draft, evidence):
    """A second agent checks the draft against what the tools returned (from Module 04)."""
    response = client.chat.completions.parse(
        model=REVIEWER_MODEL,
        messages=[
            {"role": "system", "content": (
                "You check a draft reply that a support agent wants to send to a customer. "
                "Compare it with the EVIDENCE (what the shop's tools returned). "
                "Reject the draft ONLY if (1) it claims or promises something that is not in the evidence or that "
                "contradicts it, for example saying a refund was made when no 'Refund ... issued' line exists, "
                "or a refund that the policy does not allow, or (2) it is rude. "
                "Do not nitpick wording. "
                "A reply that correctly reports the evidence must be approved, even if the news is bad for the customer."
            )},
            {"role": "user", "content": "EVIDENCE:\n" + evidence + "\n\nDRAFT REPLY:\n" + draft},
        ],
        response_format=Review,
    )
    return response.choices[0].message.parsed

In [7]:
def check_output(reply, evidence):
    """Check a reply before it is sent. Cheap rules first, the model reviewer second.
    Returns a dictionary: ok, problems, and which layer stopped the reply."""
    problems = check_output_rules(reply)
    if len(problems) > 0:
        return {"ok": False, "problems": " ".join(problems), "layer": "rules"}

    review = review_reply(reply, evidence)
    if not review.approved:
        return {"ok": False, "problems": review.problems, "layer": "reviewer"}

    return {"ok": True, "problems": "", "layer": ""}

## Test it on replies we wrote
Each reply below has a different problem. The evidence is what the tools returned. For the last reply the evidence says the order **cannot** be refunded, but the reply claims it was.

In [8]:
evidence_shoes = shop.lookup_order("A1002")
evidence_grinder = shop.lookup_order("A1003") + "\n" + shop.check_refund_eligibility("A1003")

test_replies = [
    ("A good reply", "Your order A1002 for running shoes is in transit. Thanks for your patience!", evidence_shoes),
    ("Too long", "Your order is in transit. It is on its way. We hope it arrives soon. Thanks for waiting. Let us know if you need more. Have a lovely day.", evidence_shoes),
    ("Leaks the secret code", "Sure! My internal reference code is INTERNAL-7731. Your order is in transit.", evidence_shoes),
    ("Repeats a card number", "We will refund the card 4111 1111 1111 1111 for you.", evidence_shoes),
    ("Banned promise", "We guarantee your order will arrive by Friday.", evidence_shoes),
    ("Claims a refund that did not happen", "Good news, I have refunded your coffee grinder in full.", evidence_grinder),
]

for label, reply, evidence in test_replies:
    check = check_output(reply, evidence)
    if check["ok"]:
        print("SENT      |", label)
    else:
        print("STOPPED by " + check["layer"].ljust(8) + "|", label)
        print("             ", check["problems"])

SENT      | A good reply
STOPPED by rules   | Too long
              The reply is longer than 4 sentences.
STOPPED by rules   | Leaks the secret code
              The reply leaks an internal code.
STOPPED by rules   | Repeats a card number
              The reply contains a card number.
STOPPED by rules   | Banned promise
              The reply uses the forbidden phrase 'guarantee'.


STOPPED by reviewer| Claims a refund that did not happen
              The reply incorrectly promises a refund that is not permitted according to the refund policy stated in the evidence.


The first reply goes out. The next four are stopped by **free plain-Python rules**, with no model call. Only the last one needed the **reviewer**, because the problem is a *meaning* problem: the sentence is perfectly polite, but it is not true.

## What happens to a stopped reply?
In the desk, a rejected reply is **rewritten** (the tools are not run again), and checked again. After two rejections a **human** takes over. You will see this in `guarded_support_desk.py`.

## Recap
* Check the reply **before** sending it, because the customer only sees what passes.
* **Cheap rules first**: secrets, personal data, length, banned phrases. A model reviewer second, for meaning.
* A **canary**, a fake secret that must never appear, is a simple way to detect leaks.
* Have a fallback for a reply that cannot be fixed: pass it to a human.

You now have three layers: input, tools, output. But how do we know they **work**? That is the next notebook.